# Convert row indices $S$ to `x[i,j,k]`, `y[i,j,k]`, and `z[i,j,k]`

This notebook supports arbitrary positive integers `m`, `n`, `p`, and `r`.

For each layer `k`, the global variable order is

```text
[vec(x[:,:,k]); vec(y[:,:,k]); vec(z[:,:,k])]
```

where

```text
x[:,:,k] has size m × n
y[:,:,k] has size n × p
z[:,:,k] has size p × m
```

Julia uses column-major vectorization, and the row indices in `S` are assumed to be 1-based.

The conversion now supports three outputs:

1. structured Julia records returned in memory;
2. a plain-text file with one variable name per line;
3. an Oscar-compatible `.jl` file of the form `Svars = [...]`.


In [ ]:
using DelimitedFiles

In [ ]:
# Set `m`, `n`, `p`, `r`

m = 4
n = 4
p = 4
r = 49

In [ ]:
function row_to_xyz(
    row::Integer;
    m::Integer,
    n::Integer,
    p::Integer,
    r::Integer,
)
    @assert m > 0 "m must be positive."
    @assert n > 0 "n must be positive."
    @assert p > 0 "p must be positive."
    @assert r > 0 "r must be positive."

    x_size = m * n
    y_size = n * p
    z_size = p * m
    layer_size = x_size + y_size + z_size
    total_size = r * layer_size

    @assert 1 <= row <= total_size (
        "Row $row is outside the valid range 1:$total_size."
    )

    zero_based_row = Int(row) - 1
    k = div(zero_based_row, layer_size) + 1
    local_offset = mod(zero_based_row, layer_size) + 1

    if local_offset <= x_size
        block = :x
        local_index = local_offset
        number_of_rows = m
        number_of_columns = n
    elseif local_offset <= x_size + y_size
        block = :y
        local_index = local_offset - x_size
        number_of_rows = n
        number_of_columns = p
    else
        block = :z
        local_index = local_offset - x_size - y_size
        number_of_rows = p
        number_of_columns = m
    end

    i = mod(local_index - 1, number_of_rows) + 1
    j = div(local_index - 1, number_of_rows) + 1

    @assert 1 <= j <= number_of_columns

    variable = "$(block)[$i,$j,$k]"

    return (
        row = Int(row),
        variable = variable,
        block = block,
        i = i,
        j = j,
        k = k,
        local_offset = local_offset,
        local_index = local_index,
    )
end

In [ ]:
function write_plain_variables(
    records;
    output_file::AbstractString,
)
    open(output_file, "w") do io
        for record in records
            println(io, record.variable)
        end
    end

    return abspath(output_file)
end

In [ ]:
function write_oscar_variables(
    records;
    output_file::AbstractString,
    variable_name::AbstractString = "Svars",
    include_header::Bool = true,
)
    @assert occursin(
        r"^[A-Za-z_][A-Za-z_0-9]*$",
        variable_name,
    ) "Invalid Julia variable name: $variable_name"

    open(output_file, "w") do io
        if include_header
            println(
                io,
                "# Run this file after creating the Oscar variables x, y, z.",
            )
        end

        println(io, variable_name, " = [")

        for record in records
            println(io, "    ", record.variable, ",")
        end

        println(io, "]")
    end

    return abspath(output_file)
end

In [ ]:
function S_to_XYZ(
    S;
    m::Integer,
    n::Integer,
    p::Integer,
    r::Integer,
    output_file::Union{Nothing, AbstractString} = "vars.txt",
    output_oscar_file::Union{Nothing, AbstractString} = nothing,
    oscar_variable_name::AbstractString = "Svars",
    include_oscar_header::Bool = true,
    sort_rows::Bool = true,
    remove_duplicates::Bool = true,
)
    rows = Int.(collect(S))

    if remove_duplicates
        # unique preserves the first occurrence when sort_rows=false.
        rows = unique(rows)
    end

    if sort_rows
        sort!(rows)
    end

    records = [
        row_to_xyz(
            row;
            m=m,
            n=n,
            p=p,
            r=r,
        )
        for row in rows
    ]

    println("Number of input rows: ", length(rows))

    if output_file !== nothing
        text_path = write_plain_variables(
            records;
            output_file=output_file,
        )
        println("Plain-text variables written to: ", text_path)
    end

    if output_oscar_file !== nothing
        oscar_path = write_oscar_variables(
            records;
            output_file=output_oscar_file,
            variable_name=oscar_variable_name,
            include_header=include_oscar_header,
        )
        println("Oscar-format variables written to: ", oscar_path)
    end

    return records
end

In [ ]:
function read_S_indices(filename::AbstractString)
    isfile(filename) || error("File not found: $(abspath(filename))")

    rows = Int[]

    for (line_number, line) in enumerate(eachline(filename))
        clean = strip(
            replace(
                line,
                ',' => ' ',
                '[' => ' ',
                ']' => ' ',
            ),
        )

        isempty(clean) && continue

        for token in split(clean)
            value = tryparse(Int, token)

            value === nothing && error(
                "Cannot parse token '$token' as an integer " *
                "in $filename at line $line_number.",
            )

            push!(rows, value)
        end
    end

    isempty(rows) && error("No row indices were found in $filename.")

    return rows
end

### 1.Direct input

In [ ]:
# Replace this example by your own 1-based row index set S.
S = [
    69,
    70,
    77,
    78,
    83,
    84,
    87,
    88
]

In [ ]:
records = S_to_XYZ(
    S;
    m=m,
    n=n,
    p=p,
    r=r,
    output_file="vars_test.txt",
    output_oscar_file="vars_test_oscar.jl",
    oscar_variable_name="Svars",
)

In [ ]:
getproperty.(records, :variable)

In [ ]:
# Preview the generated Oscar source file.
println(read("vars_test_oscar.jl", String))

### 2.Read `S` from a text file

Use this when a file such as `Stest.txt` contains row indices. The input may contain one integer per line, with or without trailing commas.

In [ ]:
S1 = read_S_indices("Sgap10_size83.txt")

In [ ]:
records_from_file = S_to_XYZ(
    S1;
    m=m,
    n=n,
    p=p,
    r=r,
    output_file="vars_Sgap10.txt",
    output_oscar_file="vars_Sgap10_oscar.jl",
    oscar_variable_name="Svars"
)